In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# Generate baselines

- zero-return forecast
- training-mean forecast
- training-median forecast
- conventional MASE

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_daily"
)


# ============================================================
# DATA HELPERS
# ============================================================

def load_xy(path):
    """
    Load one supervised dataset.

    X:
        shape (n_origins, 250)

    Y:
        shape (n_origins, 20)
    """

    df = pd.read_csv(path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def reconstruct_training_series(train_path):
    """
    Reconstruct the unique chronological return series from
    overlapping supervised windows.

    With LOOKBACK=250, HORIZON=20, SLIDE=1:

        row 0:
            X = t0 ... t249
            Y = t250 ... t269

        row 1:
            X = t1 ... t250
            Y = t251 ... t270

    Therefore:
        - the first row provides t0 ... t269
        - each later row contributes exactly one new observation:
          its final target Y_19
    """

    X, Y = load_xy(train_path)

    if len(X) == 0:
        raise ValueError(
            f"Training file is empty: {train_path}"
        )

    # Complete first supervised window:
    # X_0 ... X_249, then Y_0 ... Y_19
    first_window = np.concatenate([
        X[0],
        Y[0],
    ])

    # Because slide = 1, every later row contributes
    # one new observation at its far-right endpoint.
    if len(Y) > 1:
        later_points = Y[1:, -1]

        train_series = np.concatenate([
            first_window,
            later_points,
        ])
    else:
        train_series = first_window

    return train_series


# ============================================================
# CONVENTIONAL MASE SCALE
# ============================================================

def calculate_mase_scale(train_series):
    """
    Conventional MASE denominator based only on the
    chronological training series.

    Scale =
        mean absolute one-step naive error in training data

        mean(|y_t - y_{t-1}|)

    This denominator is fixed for all models and forecast
    horizons belonging to the same dataset.
    """

    train_series = np.asarray(
        train_series,
        dtype=float,
    )

    if len(train_series) < 2:
        raise ValueError(
            "Need at least two training observations "
            "to calculate MASE scale."
        )

    naive_errors = np.abs(
        train_series[1:]
        - train_series[:-1]
    )

    scale = float(
        np.mean(naive_errors)
    )

    if not np.isfinite(scale):
        raise ValueError(
            "MASE scale is not finite."
        )

    if np.isclose(scale, 0.0):
        raise ValueError(
            "MASE scale is zero."
        )

    return scale


# ============================================================
# BASELINE FORECASTS
# ============================================================

def make_baseline_forecasts(
    X_test,
    train_series,
):
    """
    Construct simple point-forecast baselines.

    Returned forecasts all have shape:

        (n_test_origins, HORIZON)
    """

    n_origins = X_test.shape[0]

    # --------------------------------------------------------
    # 1. Zero-return forecast
    # --------------------------------------------------------

    zero_forecast = np.zeros(
        (n_origins, HORIZON),
        dtype=float,
    )

    # --------------------------------------------------------
    # 2. Training-mean forecast
    #
    # Same training-series mean at every origin/horizon.
    # Appropriate simple location baseline for squared error.
    # --------------------------------------------------------

    train_mean = float(
        np.mean(train_series)
    )

    train_mean_forecast = np.full(
        (n_origins, HORIZON),
        train_mean,
        dtype=float,
    )

    # --------------------------------------------------------
    # 3. Training-median forecast
    #
    # Same training-series median at every origin/horizon.
    # Appropriate simple location baseline for absolute error.
    # --------------------------------------------------------

    train_median = float(
        np.median(train_series)
    )

    train_median_forecast = np.full(
        (n_origins, HORIZON),
        train_median,
        dtype=float,
    )

    # --------------------------------------------------------
    # 4. Persistence forecast
    #
    # Last observed return in the 250-point test context,
    # repeated across all future horizons.
    # --------------------------------------------------------

    last_observation = X_test[:, -1]

    persistence_forecast = np.repeat(
        last_observation[:, None],
        HORIZON,
        axis=1,
    )

    return {
        "Zero": zero_forecast,
        "Train Mean": train_mean_forecast,
        "Train Median": train_median_forecast,
        "Persistence": persistence_forecast,
    }


# ============================================================
# METRICS
# ============================================================

def calculate_point_metrics(
    Y,
    forecast,
    mase_scale,
):
    """
    Calculate overall RMSE, MAE and conventional MASE.

    Overall metrics aggregate over all forecast origins
    and all 20 horizons.
    """

    Y = np.asarray(Y, dtype=float)
    forecast = np.asarray(
        forecast,
        dtype=float,
    )

    if Y.shape != forecast.shape:
        raise ValueError(
            f"Y shape {Y.shape} does not match "
            f"forecast shape {forecast.shape}."
        )

    error = Y - forecast

    # RMSE for each of the 20 horizons
    rmse_by_horizon = np.sqrt(
        np.mean(
            error ** 2,
            axis=0,
        )
    )
    
    # Overall RMSE: average the horizon-specific RMSEs
    rmse = float(
        np.mean(rmse_by_horizon)
    )

    mae = float(
        np.mean(
            np.abs(error)
        )
    )

    mase = float(
        mae / mase_scale
    )

    # --------------------------------------------------------
    # Horizon-specific metrics
    # --------------------------------------------------------

    # rmse_by_horizon = np.sqrt(
    #     np.mean(
    #         error ** 2,
    #         axis=0,
    #     )
    # )

    mae_by_horizon = np.mean(
        np.abs(error),
        axis=0,
    )

    mase_by_horizon = (
        mae_by_horizon
        / mase_scale
    )

    return {
        "rmse": rmse,
        "mae": mae,
        "mase": mase,
        "rmse_by_horizon": rmse_by_horizon,
        "mae_by_horizon": mae_by_horizon,
        "mase_by_horizon": mase_by_horizon,
    }


# ============================================================
# CALCULATE BASELINES FOR ALL DAILY DATASETS
# ============================================================

summary_rows = []
horizon_rows = []


for asset_class, asset_name in ASSET_ORDER.items():

    print()
    print("=" * 70)
    print(f"ASSET CLASS: {asset_name}")
    print("=" * 70)

    asset_dir = DATA_DIR / asset_class

    test_files = sorted(
        asset_dir.glob("test_*.csv")
    )

    for test_path in test_files:

        try:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            train_path = (
                asset_dir
                / f"train_{dataset_name}.csv"
            )

            if not train_path.exists():
                raise FileNotFoundError(
                    f"Training file not found: "
                    f"{train_path}"
                )

            # ------------------------------------------------
            # Load held-out test data
            # ------------------------------------------------

            X_test, Y_test = load_xy(
                test_path
            )

            # ------------------------------------------------
            # Reconstruct chronological training series
            # ------------------------------------------------

            train_series = (
                reconstruct_training_series(
                    train_path
                )
            )

            # ------------------------------------------------
            # Conventional MASE denominator
            # ------------------------------------------------

            mase_scale = calculate_mase_scale(
                train_series
            )

            # ------------------------------------------------
            # Baselines
            # ------------------------------------------------

            forecasts = make_baseline_forecasts(
                X_test=X_test,
                train_series=train_series,
            )

            # ------------------------------------------------
            # Evaluate each baseline
            # ------------------------------------------------

            for baseline_name, forecast in forecasts.items():

                result = calculate_point_metrics(
                    Y=Y_test,
                    forecast=forecast,
                    mase_scale=mase_scale,
                )

                summary_rows.append({
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "baseline": baseline_name,
                    "rmse": result["rmse"],
                    "mae": result["mae"],
                    "mase": result["mase"],
                    "mase_scale": mase_scale,
                    "train_mean": float(
                        np.mean(train_series)
                    ),
                    "train_median": float(
                        np.median(train_series)
                    ),
                    "n_train_observations": (
                        len(train_series)
                    ),
                    "n_test_origins": (
                        len(Y_test)
                    ),
                })

                for h in range(HORIZON):

                    horizon_rows.append({
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "baseline": baseline_name,
                        "horizon": h + 1,
                        "rmse": (
                            result[
                                "rmse_by_horizon"
                            ][h]
                        ),
                        "mae": (
                            result[
                                "mae_by_horizon"
                            ][h]
                        ),
                        "mase": (
                            result[
                                "mase_by_horizon"
                            ][h]
                        ),
                        "mase_scale": mase_scale,
                    })

            print(
                f"  {dataset_name}: "
                f"MASE scale={mase_scale:.6f}"
            )

        except Exception as exc:

            print(
                f"  ERROR {test_path.name}: "
                f"{exc}"
            )


# ============================================================
# SAVE RESULTS
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

horizon_df = pd.DataFrame(
    horizon_rows
)


summary_path = (
    BASE_DIR
    / "point_baselines_daily.csv"
)

horizon_path = (
    BASE_DIR
    / "point_baselines_by_horizon_daily.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)

horizon_df.to_csv(
    horizon_path,
    index=False,
)


print()
print("=" * 70)
print("SAVED")
print("=" * 70)

print(summary_path)
print(horizon_path)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# WEEKLY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_weekly"
)


# ============================================================
# DATA HELPERS
# ============================================================

def load_xy(path):
    """
    Load one supervised dataset.

    X:
        shape (n_origins, 250)

    Y:
        shape (n_origins, 20)
    """

    df = pd.read_csv(path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def reconstruct_training_series(train_path):
    """
    Reconstruct the unique chronological return series from
    overlapping supervised windows.

    With LOOKBACK=250, HORIZON=20, SLIDE=1:

        row 0:
            X = t0 ... t249
            Y = t250 ... t269

        row 1:
            X = t1 ... t250
            Y = t251 ... t270

    Therefore:
        - the first row provides t0 ... t269
        - each later row contributes exactly one new observation:
          its final target Y_19
    """

    X, Y = load_xy(train_path)

    if len(X) == 0:
        raise ValueError(
            f"Training file is empty: {train_path}"
        )

    # Complete first supervised window:
    # X_0 ... X_249, then Y_0 ... Y_19
    first_window = np.concatenate([
        X[0],
        Y[0],
    ])

    # Because slide = 1, every later row contributes
    # one new observation at its far-right endpoint.
    if len(Y) > 1:
        later_points = Y[1:, -1]

        train_series = np.concatenate([
            first_window,
            later_points,
        ])
    else:
        train_series = first_window

    return train_series


# ============================================================
# CONVENTIONAL MASE SCALE
# ============================================================

def calculate_mase_scale(train_series):
    """
    Conventional MASE denominator based only on the
    chronological training series.

    Scale =
        mean absolute one-step naive error in training data

        mean(|y_t - y_{t-1}|)

    This denominator is fixed for all models and forecast
    horizons belonging to the same dataset.
    """

    train_series = np.asarray(
        train_series,
        dtype=float,
    )

    if len(train_series) < 2:
        raise ValueError(
            "Need at least two training observations "
            "to calculate MASE scale."
        )

    naive_errors = np.abs(
        train_series[1:]
        - train_series[:-1]
    )

    scale = float(
        np.mean(naive_errors)
    )

    if not np.isfinite(scale):
        raise ValueError(
            "MASE scale is not finite."
        )

    if np.isclose(scale, 0.0):
        raise ValueError(
            "MASE scale is zero."
        )

    return scale


# ============================================================
# BASELINE FORECASTS
# ============================================================

def make_baseline_forecasts(
    X_test,
    train_series,
):
    """
    Construct simple point-forecast baselines.

    Returned forecasts all have shape:

        (n_test_origins, HORIZON)
    """

    n_origins = X_test.shape[0]

    # --------------------------------------------------------
    # 1. Zero-return forecast
    # --------------------------------------------------------

    zero_forecast = np.zeros(
        (n_origins, HORIZON),
        dtype=float,
    )

    # --------------------------------------------------------
    # 2. Training-mean forecast
    #
    # Same training-series mean at every origin/horizon.
    # Appropriate simple location baseline for squared error.
    # --------------------------------------------------------

    train_mean = float(
        np.mean(train_series)
    )

    train_mean_forecast = np.full(
        (n_origins, HORIZON),
        train_mean,
        dtype=float,
    )

    # --------------------------------------------------------
    # 3. Training-median forecast
    #
    # Same training-series median at every origin/horizon.
    # Appropriate simple location baseline for absolute error.
    # --------------------------------------------------------

    train_median = float(
        np.median(train_series)
    )

    train_median_forecast = np.full(
        (n_origins, HORIZON),
        train_median,
        dtype=float,
    )

    # --------------------------------------------------------
    # 4. Persistence forecast
    #
    # Last observed return in the 250-point test context,
    # repeated across all future horizons.
    # --------------------------------------------------------

    last_observation = X_test[:, -1]

    persistence_forecast = np.repeat(
        last_observation[:, None],
        HORIZON,
        axis=1,
    )

    return {
        "Zero": zero_forecast,
        "Train Mean": train_mean_forecast,
        "Train Median": train_median_forecast,
        "Persistence": persistence_forecast,
    }


# ============================================================
# METRICS
# ============================================================

def calculate_point_metrics(
    Y,
    forecast,
    mase_scale,
):
    """
    Calculate overall RMSE, MAE and conventional MASE.

    Overall metrics aggregate over all forecast origins
    and all 20 horizons.
    """

    Y = np.asarray(Y, dtype=float)
    forecast = np.asarray(
        forecast,
        dtype=float,
    )

    if Y.shape != forecast.shape:
        raise ValueError(
            f"Y shape {Y.shape} does not match "
            f"forecast shape {forecast.shape}."
        )

    error = Y - forecast

    # RMSE for each of the 20 horizons
    rmse_by_horizon = np.sqrt(
        np.mean(
            error ** 2,
            axis=0,
        )
    )
    
    # Overall RMSE: average the horizon-specific RMSEs
    rmse = float(
        np.mean(rmse_by_horizon)
    )

    mae = float(
        np.mean(
            np.abs(error)
        )
    )

    mase = float(
        mae / mase_scale
    )

    # --------------------------------------------------------
    # Horizon-specific metrics
    # --------------------------------------------------------

    # rmse_by_horizon = np.sqrt(
    #     np.mean(
    #         error ** 2,
    #         axis=0,
    #     )
    # )

    mae_by_horizon = np.mean(
        np.abs(error),
        axis=0,
    )

    mase_by_horizon = (
        mae_by_horizon
        / mase_scale
    )

    return {
        "rmse": rmse,
        "mae": mae,
        "mase": mase,
        "rmse_by_horizon": rmse_by_horizon,
        "mae_by_horizon": mae_by_horizon,
        "mase_by_horizon": mase_by_horizon,
    }


# ============================================================
# CALCULATE BASELINES FOR ALL DAILY DATASETS
# ============================================================

summary_rows = []
horizon_rows = []


for asset_class, asset_name in ASSET_ORDER.items():

    print()
    print("=" * 70)
    print(f"ASSET CLASS: {asset_name}")
    print("=" * 70)

    asset_dir = DATA_DIR / asset_class

    test_files = sorted(
        asset_dir.glob("test_*.csv")
    )

    for test_path in test_files:

        try:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            train_path = (
                asset_dir
                / f"train_{dataset_name}.csv"
            )

            if not train_path.exists():
                raise FileNotFoundError(
                    f"Training file not found: "
                    f"{train_path}"
                )

            # ------------------------------------------------
            # Load held-out test data
            # ------------------------------------------------

            X_test, Y_test = load_xy(
                test_path
            )

            # ------------------------------------------------
            # Reconstruct chronological training series
            # ------------------------------------------------

            train_series = (
                reconstruct_training_series(
                    train_path
                )
            )

            # ------------------------------------------------
            # Conventional MASE denominator
            # ------------------------------------------------

            mase_scale = calculate_mase_scale(
                train_series
            )

            # ------------------------------------------------
            # Baselines
            # ------------------------------------------------

            forecasts = make_baseline_forecasts(
                X_test=X_test,
                train_series=train_series,
            )

            # ------------------------------------------------
            # Evaluate each baseline
            # ------------------------------------------------

            for baseline_name, forecast in forecasts.items():

                result = calculate_point_metrics(
                    Y=Y_test,
                    forecast=forecast,
                    mase_scale=mase_scale,
                )

                summary_rows.append({
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "baseline": baseline_name,
                    "rmse": result["rmse"],
                    "mae": result["mae"],
                    "mase": result["mase"],
                    "mase_scale": mase_scale,
                    "train_mean": float(
                        np.mean(train_series)
                    ),
                    "train_median": float(
                        np.median(train_series)
                    ),
                    "n_train_observations": (
                        len(train_series)
                    ),
                    "n_test_origins": (
                        len(Y_test)
                    ),
                })

                for h in range(HORIZON):

                    horizon_rows.append({
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "baseline": baseline_name,
                        "horizon": h + 1,
                        "rmse": (
                            result[
                                "rmse_by_horizon"
                            ][h]
                        ),
                        "mae": (
                            result[
                                "mae_by_horizon"
                            ][h]
                        ),
                        "mase": (
                            result[
                                "mase_by_horizon"
                            ][h]
                        ),
                        "mase_scale": mase_scale,
                    })

            print(
                f"  {dataset_name}: "
                f"MASE scale={mase_scale:.6f}"
            )

        except Exception as exc:

            print(
                f"  ERROR {test_path.name}: "
                f"{exc}"
            )


# ============================================================
# SAVE RESULTS
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

horizon_df = pd.DataFrame(
    horizon_rows
)


summary_path = (
    BASE_DIR
    / "point_baselines_weekly.csv"
)

horizon_path = (
    BASE_DIR
    / "point_baselines_by_horizon_weekly.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)

horizon_df.to_csv(
    horizon_path,
    index=False,
)


print()
print("=" * 70)
print("SAVED")
print("=" * 70)

print(summary_path)
print(horizon_path)

# Compute conventional MASE

In [ ]:
import pandas as pd
from pathlib import Path


BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)


# ============================================================
# LOAD EXISTING MODEL MAE
# ============================================================

mae_df = pd.read_csv(
    BASE_DIR / "mae_summary_all_models.csv"
)


# ============================================================
# LOAD DATASET-SPECIFIC MASE SCALES
# ============================================================

baseline_df = pd.read_csv(
    BASE_DIR / "point_baselines_daily.csv"
)

mase_scales = (
    baseline_df[
        [
            "asset_class",
            "dataset",
            "mase_scale",
        ]
    ]
    .drop_duplicates()
)


# ============================================================
# MERGE
# ============================================================

out = mae_df.merge(
    mase_scales,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)


if out["mase_scale"].isna().any():
    missing = out[
        out["mase_scale"].isna()
    ][
        [
            "asset_class",
            "dataset",
        ]
    ].drop_duplicates()

    raise ValueError(
        "Missing MASE scales for:\n"
        + missing.to_string(index=False)
    )


# ============================================================
# CONVENTIONAL MASE
# ============================================================

out["mase"] = (
    out["mae"]
    / out["mase_scale"]
)


# ============================================================
# SAVE
# ============================================================

output_path = (
    BASE_DIR
    / "mase_conventional_summary_all_models.csv"
)

out.to_csv(
    output_path,
    index=False,
)

print(out.head())
print()
print(f"Rows: {len(out)}")
print(f"Saved to: {output_path}")

In [ ]:
import pandas as pd
from pathlib import Path


BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

mae_h = pd.read_csv(
    BASE_DIR / "mae_by_horizon_all_models.csv"
)

out_h = mae_h.merge(
    mase_scales,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

out_h["mase"] = (
    out_h["mae"]
    / out_h["mase_scale"]
)

out_h.to_csv(
    BASE_DIR / "mase_conventional_by_horizon_all_models.csv",
    index=False,
)

print(out_h.head())
print()
print(f"Rows: {len(out_h)}")

In [ ]:
import pandas as pd
from pathlib import Path


BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)


# ============================================================
# LOAD EXISTING MODEL MAE
# ============================================================

mae_df = pd.read_csv(
    BASE_DIR / "mae_summary_all_models.csv"
)


# ============================================================
# LOAD DATASET-SPECIFIC MASE SCALES
# ============================================================

baseline_df = pd.read_csv(
    BASE_DIR / "point_baselines_weekly.csv"
)

mase_scales = (
    baseline_df[
        [
            "asset_class",
            "dataset",
            "mase_scale",
        ]
    ]
    .drop_duplicates()
)


# ============================================================
# MERGE
# ============================================================

out = mae_df.merge(
    mase_scales,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)


if out["mase_scale"].isna().any():
    missing = out[
        out["mase_scale"].isna()
    ][
        [
            "asset_class",
            "dataset",
        ]
    ].drop_duplicates()

    raise ValueError(
        "Missing MASE scales for:\n"
        + missing.to_string(index=False)
    )


# ============================================================
# CONVENTIONAL MASE
# ============================================================

out["mase"] = (
    out["mae"]
    / out["mase_scale"]
)


# ============================================================
# SAVE
# ============================================================

output_path = (
    BASE_DIR
    / "mase_conventional_summary_all_models.csv"
)

out.to_csv(
    output_path,
    index=False,
)

print(out.head())
print()
print(f"Rows: {len(out)}")
print(f"Saved to: {output_path}")

In [ ]:
import pandas as pd
from pathlib import Path


BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

mae_h = pd.read_csv(
    BASE_DIR / "mae_by_horizon_all_models.csv"
)

out_h = mae_h.merge(
    mase_scales,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

out_h["mase"] = (
    out_h["mae"]
    / out_h["mase_scale"]
)

out_h.to_csv(
    BASE_DIR / "mase_conventional_by_horizon_all_models.csv",
    index=False,
)

print(out_h.head())
print()
print(f"Rows: {len(out_h)}")

# Comparisons

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

MODEL_MASE_PATH = (
    BASE_DIR
    / "mase_conventional_summary_all_models.csv"
)

BASELINE_PATH = (
    BASE_DIR
    / "point_baselines_daily.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "mase_robustness_table_daily.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

models = pd.read_csv(MODEL_MASE_PATH)
baselines = pd.read_csv(BASELINE_PATH)


# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

# Existing model file should contain:
# model | asset_class | dataset | mae | mase_scale | mase

model_df = models[
    [
        "model",
        "asset_class",
        "dataset",
        "mase",
    ]
].copy()

model_df = model_df.rename(
    columns={
        "model": "method",
    }
)


# Baseline file should contain:
# asset_class | dataset | baseline | rmse | mae | mase | ...

baseline_df = baselines[
    [
        "baseline",
        "asset_class",
        "dataset",
        "mase",
    ]
].copy()

baseline_df = baseline_df.rename(
    columns={
        "baseline": "method",
    }
)


# ============================================================
# COMBINE MODELS + BASELINES
# ============================================================

all_results = pd.concat(
    [
        model_df,
        baseline_df,
    ],
    ignore_index=True,
)


# ============================================================
# EXTRACT ZERO AND TRAIN-MEDIAN REFERENCES
# ============================================================

zero_ref = (
    baseline_df[
        baseline_df["method"] == "Zero"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "zero_mase",
        }
    )
)

median_ref = (
    baseline_df[
        baseline_df["method"] == "Train Median"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "train_median_mase",
        }
    )
)

mean_ref = (
    baseline_df[
        baseline_df["method"] == "Train Mean"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "train_mean_mase",
        }
    )
)


# ============================================================
# MERGE REFERENCES ONTO EVERY METHOD
# ============================================================

all_results = all_results.merge(
    zero_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

all_results = all_results.merge(
    median_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

all_results = all_results.merge(
    mean_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

# ============================================================
# CHECK FOR MISSING REFERENCES
# ============================================================

if all_results["zero_mase"].isna().any():
    raise ValueError(
        "Missing zero baseline for some datasets."
    )

if all_results["train_median_mase"].isna().any():
    raise ValueError(
        "Missing training-median baseline for some datasets."
    )


# ============================================================
# DATASET-LEVEL INDICATORS
# ============================================================

all_results["mase_below_1"] = (
    all_results["mase"] < 1.0
)

all_results["beats_zero"] = (
    all_results["mase"]
    < all_results["zero_mase"]
)

all_results["beats_train_median"] = (
    all_results["mase"]
    < all_results["train_median_mase"]
)

all_results["beats_train_mean"] = (
    all_results["mase"]
    < all_results["train_mean_mase"]
)

# ============================================================
# AGGREGATE BY METHOD
# ============================================================

table = (
    all_results
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        mean_mase=(
            "mase",
            "mean",
        ),
        median_mase=(
            "mase",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_mase_below_1=(
            "mase_below_1",
            "sum",
        ),
        n_beats_zero=(
            "beats_zero",
            "sum",
        ),
        n_beats_train_median=(
            "beats_train_median",
            "sum",
        ),
        n_beats_train_mean=("beats_train_mean", "sum"),
    )
)


# ============================================================
# CONVERT COUNTS TO PERCENTAGES
# ============================================================

table["pct_mase_below_1"] = (
    100
    * table["n_mase_below_1"]
    / table["n_datasets"]
)

table["pct_beats_zero"] = (
    100
    * table["n_beats_zero"]
    / table["n_datasets"]
)

table["pct_beats_train_median"] = (
    100
    * table["n_beats_train_median"]
    / table["n_datasets"]
)

table["pct_beats_train_mean"] = (
    100
    * table["n_beats_train_mean"]
    / table["n_datasets"]
)

# ============================================================
# MAKE SELF-COMPARISONS BLANK
# ============================================================

table.loc[
    table["method"] == "Zero",
    [
        "n_beats_zero",
        "pct_beats_zero",
    ],
] = pd.NA

table.loc[
    table["method"] == "Train Median",
    [
        "n_beats_train_median",
        "pct_beats_train_median",
    ],
] = pd.NA

table.loc[
    table["method"] == "Train Mean",
    [
        "n_beats_train_mean",
        "pct_beats_train_mean",
    ],
] = pd.NA

# ============================================================
# SORT BY MEAN MASE
# ============================================================

table = table.sort_values(
    "mean_mase",
    ascending=True,
).reset_index(drop=True)


# ============================================================
# ROUND DISPLAY VALUES
# ============================================================

table["mean_mase"] = (
    table["mean_mase"]
    .round(3)
)

table["median_mase"] = (
    table["median_mase"]
    .round(3)
)

table["pct_mase_below_1"] = (
    table["pct_mase_below_1"]
    .round(1)
)

table["pct_beats_zero"] = (
    table["pct_beats_zero"]
    .round(1)
)

table["pct_beats_train_median"] = (
    table["pct_beats_train_median"]
    .round(1)
)

table["pct_beats_train_mean"] = (
    table["pct_beats_train_mean"]
    .round(1)
)

# ============================================================
# SELECT FINAL TABLE COLUMNS
# ============================================================

final_table = table[
    [
        "method",
        "mean_mase",
        "median_mase",
        "pct_mase_below_1",
        "pct_beats_zero",
        "pct_beats_train_median",
        "pct_beats_train_mean",
        "n_beats_zero",
        "n_beats_train_median",
        "n_beats_train_mean"
    ]
].copy()


# ============================================================
# RENAME FOR PRESENTATION
# ============================================================

final_table = final_table.rename(
    columns={
        "method": "Method",
        "mean_mase": "Mean MASE",
        "median_mase": "Median MASE",
        "pct_mase_below_1": "MASE < 1 (%)",
        "pct_beats_zero": "Beats Zero (%)",
        "n_beats_zero": "Beats Zero (Count)",
        "pct_beats_train_median": (
            "Beats Train Median (%)"
        ),
        "n_beats_train_median": "Beats Train Median (Count)",
        "pct_beats_train_mean": (
            "Beats Train Mean (%)"
        ),
        "n_beats_train_mean": "Beats Train Mean (Count)",
    }
)


# ============================================================
# SAVE + PRINT
# ============================================================

final_table.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(final_table.to_string(index=False))
print()
print(f"Saved to: {OUTPUT_PATH}")

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

MODEL_MASE_PATH = (
    BASE_DIR
    / "mase_conventional_summary_all_models.csv"
)

BASELINE_PATH = (
    BASE_DIR
    / "point_baselines_weekly.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "mase_robustness_table_weekly.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

models = pd.read_csv(MODEL_MASE_PATH)
baselines = pd.read_csv(BASELINE_PATH)


# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

# Existing model file should contain:
# model | asset_class | dataset | mae | mase_scale | mase

model_df = models[
    [
        "model",
        "asset_class",
        "dataset",
        "mase",
    ]
].copy()

model_df = model_df.rename(
    columns={
        "model": "method",
    }
)


# Baseline file should contain:
# asset_class | dataset | baseline | rmse | mae | mase | ...

baseline_df = baselines[
    [
        "baseline",
        "asset_class",
        "dataset",
        "mase",
    ]
].copy()

baseline_df = baseline_df.rename(
    columns={
        "baseline": "method",
    }
)


# ============================================================
# COMBINE MODELS + BASELINES
# ============================================================

all_results = pd.concat(
    [
        model_df,
        baseline_df,
    ],
    ignore_index=True,
)


# ============================================================
# EXTRACT ZERO AND TRAIN-MEDIAN REFERENCES
# ============================================================

zero_ref = (
    baseline_df[
        baseline_df["method"] == "Zero"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "zero_mase",
        }
    )
)

median_ref = (
    baseline_df[
        baseline_df["method"] == "Train Median"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "train_median_mase",
        }
    )
)

mean_ref = (
    baseline_df[
        baseline_df["method"] == "Train Mean"
    ][
        [
            "asset_class",
            "dataset",
            "mase",
        ]
    ]
    .rename(
        columns={
            "mase": "train_mean_mase",
        }
    )
)


# ============================================================
# MERGE REFERENCES ONTO EVERY METHOD
# ============================================================

all_results = all_results.merge(
    zero_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

all_results = all_results.merge(
    median_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

all_results = all_results.merge(
    mean_ref,
    on=[
        "asset_class",
        "dataset",
    ],
    how="left",
    validate="many_to_one",
)

# ============================================================
# CHECK FOR MISSING REFERENCES
# ============================================================

if all_results["zero_mase"].isna().any():
    raise ValueError(
        "Missing zero baseline for some datasets."
    )

if all_results["train_median_mase"].isna().any():
    raise ValueError(
        "Missing training-median baseline for some datasets."
    )


# ============================================================
# DATASET-LEVEL INDICATORS
# ============================================================

all_results["mase_below_1"] = (
    all_results["mase"] < 1.0
)

all_results["beats_zero"] = (
    all_results["mase"]
    < all_results["zero_mase"]
)

all_results["beats_train_median"] = (
    all_results["mase"]
    < all_results["train_median_mase"]
)

all_results["beats_train_mean"] = (
    all_results["mase"]
    < all_results["train_mean_mase"]
)

# ============================================================
# AGGREGATE BY METHOD
# ============================================================

table = (
    all_results
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        mean_mase=(
            "mase",
            "mean",
        ),
        median_mase=(
            "mase",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_mase_below_1=(
            "mase_below_1",
            "sum",
        ),
        n_beats_zero=(
            "beats_zero",
            "sum",
        ),
        n_beats_train_median=(
            "beats_train_median",
            "sum",
        ),
        n_beats_train_mean=("beats_train_mean", "sum"),
    )
)


# ============================================================
# CONVERT COUNTS TO PERCENTAGES
# ============================================================

table["pct_mase_below_1"] = (
    100
    * table["n_mase_below_1"]
    / table["n_datasets"]
)

table["pct_beats_zero"] = (
    100
    * table["n_beats_zero"]
    / table["n_datasets"]
)

table["pct_beats_train_median"] = (
    100
    * table["n_beats_train_median"]
    / table["n_datasets"]
)

table["pct_beats_train_mean"] = (
    100
    * table["n_beats_train_mean"]
    / table["n_datasets"]
)

# ============================================================
# OPTIONAL:
# MAKE SELF-COMPARISONS BLANK
# ============================================================

table.loc[
    table["method"] == "Zero",
    [
        "n_beats_zero",
        "pct_beats_zero",
    ],
] = pd.NA

table.loc[
    table["method"] == "Train Median",
    [
        "n_beats_train_median",
        "pct_beats_train_median",
    ],
] = pd.NA

table.loc[
    table["method"] == "Train Mean",
    [
        "n_beats_train_mean",
        "pct_beats_train_mean",
    ],
] = pd.NA

# ============================================================
# SORT BY MEAN MASE
# ============================================================

table = table.sort_values(
    "mean_mase",
    ascending=True,
).reset_index(drop=True)


# ============================================================
# ROUND DISPLAY VALUES
# ============================================================

table["mean_mase"] = (
    table["mean_mase"]
    .round(3)
)

table["median_mase"] = (
    table["median_mase"]
    .round(3)
)

table["pct_mase_below_1"] = (
    table["pct_mase_below_1"]
    .round(1)
)

table["pct_beats_zero"] = (
    table["pct_beats_zero"]
    .round(1)
)

table["pct_beats_train_median"] = (
    table["pct_beats_train_median"]
    .round(1)
)

table["pct_beats_train_mean"] = (
    table["pct_beats_train_mean"]
    .round(1)
)

# ============================================================
# SELECT FINAL TABLE COLUMNS
# ============================================================

final_table = table[
    [
        "method",
        "mean_mase",
        "median_mase",
        "pct_mase_below_1",
        "pct_beats_zero",
        "pct_beats_train_median",
        "pct_beats_train_mean",
        "n_beats_zero",
        "n_beats_train_median",
        "n_beats_train_mean"
    ]
].copy()


# ============================================================
# RENAME FOR PRESENTATION
# ============================================================

final_table = final_table.rename(
    columns={
        "method": "Method",
        "mean_mase": "Mean MASE",
        "median_mase": "Median MASE",
        "pct_mase_below_1": "MASE < 1 (%)",
        "pct_beats_zero": "Beats Zero (%)",
        "n_beats_zero": "Beats Zero (Count)",
        "pct_beats_train_median": (
            "Beats Train Median (%)"
        ),
        "n_beats_train_median": "Beats Train Median (Count)",
        "pct_beats_train_mean": (
            "Beats Train Mean (%)"
        ),
        "n_beats_train_mean": "Beats Train Mean (Count)",
    }
)


# ============================================================
# SAVE + PRINT
# ============================================================

final_table.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(final_table.to_string(index=False))
print()
print(f"Saved to: {OUTPUT_PATH}")

# Rolling-context conventional MASE

### Simple baselines: Zero, training median, training mean, rolling mean, rolling median, persistence

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ============================================================
# ROLLING-CONTEXT MASE SCALE
# ============================================================

def calculate_context_mase_scales(X):
    """
    X shape:
        (n_origins, 250)

    For each origin, calculate the mean absolute
    one-step naive error over its 250 observations.

    Returns:
        (n_origins,)
    """

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != LOOKBACK:
        raise ValueError(
            f"Expected X shape (n_origins, {LOOKBACK}), "
            f"got {X.shape}"
        )

    differences = np.abs(
        np.diff(X, axis=1)
    )

    scales = np.mean(
        differences,
        axis=1
    )

    if not np.all(np.isfinite(scales)):
        raise ValueError(
            "Non-finite context MASE scales detected."
        )

    if np.any(scales <= 0):
        raise ValueError(
            "Zero or negative context MASE scale detected."
        )

    return scales


# ============================================================
# SIX BASELINE FORECASTS
# ============================================================

def make_context_baselines(X_test, train_series):

    n_origins = X_test.shape[0]

    # 1. Zero return
    zero = np.zeros(
        (n_origins, HORIZON)
    )

    # 2. Fixed training mean
    train_mean = np.full(
        (n_origins, HORIZON),
        np.mean(train_series)
    )

    # 3. Fixed training median
    train_median = np.full(
        (n_origins, HORIZON),
        np.median(train_series)
    )

    # 4. Rolling-context mean
    context_means = np.mean(
        X_test,
        axis=1
    )

    context_mean = np.repeat(
        context_means[:, None],
        HORIZON,
        axis=1
    )

    # 5. Rolling-context median
    context_medians = np.median(
        X_test,
        axis=1
    )

    context_median = np.repeat(
        context_medians[:, None],
        HORIZON,
        axis=1
    )

    # 6. Last-return persistence
    persistence = np.repeat(
        X_test[:, -1, None],
        HORIZON,
        axis=1
    )

    return {
        "Zero": zero,
        "Train Mean": train_mean,
        "Train Median": train_median,
        "Context Mean": context_mean,
        "Context Median": context_median,
        "Persistence": persistence,
    }


# ============================================================
# EVALUATION
# ============================================================

def evaluate_context_mase(Y, forecast, scales):

    Y = np.asarray(Y, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    scales = np.asarray(scales, dtype=float)

    if Y.shape != forecast.shape:
        raise ValueError(
            f"Shape mismatch: {Y.shape}, {forecast.shape}"
        )

    if len(scales) != len(Y):
        raise ValueError(
            "Number of scales does not match origins."
        )

    error = Y - forecast

    # ---------------------------------------------
    # RMSE: original paper aggregation
    # ---------------------------------------------

    rmse_by_horizon = np.sqrt(
        np.mean(error ** 2, axis=0)
    )

    rmse = float(
        np.mean(rmse_by_horizon)
    )

    # ---------------------------------------------
    # MAE
    # ---------------------------------------------

    mae_by_horizon = np.mean(
        np.abs(error),
        axis=0
    )

    mae = float(
        np.mean(mae_by_horizon)
    )

    # ---------------------------------------------
    # ROLLING 250-CONTEXT MASE
    # ---------------------------------------------

    scaled_errors = (
        np.abs(error)
        / scales[:, None]
    )

    mase_by_horizon = np.mean(
        scaled_errors,
        axis=0
    )

    mase = float(
        np.mean(mase_by_horizon)
    )

    return {
        "rmse": rmse,
        "mae": mae,
        "mase_250": mase,
        "rmse_by_horizon": rmse_by_horizon,
        "mae_by_horizon": mae_by_horizon,
        "mase_250_by_horizon": mase_by_horizon,
    }


# ============================================================
# PROCESS DAILY AND WEEKLY
# ============================================================

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 70)
    print(f"PROCESSING {frequency.upper()}")
    print("=" * 70)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    if not DATA_DIR.exists():
        raise FileNotFoundError(DATA_DIR)

    if not BASE_DIR.exists():
        raise FileNotFoundError(BASE_DIR)

    summary_rows = []
    horizon_rows = []
    failures = []

    for folder, asset_class in ASSET_ORDER.items():

        asset_dir = DATA_DIR / folder

        for test_path in sorted(
            asset_dir.glob("test_*.csv")
        ):

            dataset = (
                test_path.stem
                .removeprefix("test_")
            )

            train_path = (
                asset_dir
                / f"train_{dataset}.csv"
            )

            try:

                if not train_path.exists():
                    raise FileNotFoundError(train_path)

                # Load test windows
                X_test, Y_test = load_xy(
                    test_path
                )

                # Reconstruct training series
                train_series = reconstruct_training_series(
                    train_path
                )

                # Calculate origin-specific scales
                scales = calculate_context_mase_scales(
                    X_test
                )

                # Generate six baselines
                forecasts = make_context_baselines(
                    X_test,
                    train_series
                )

                # Evaluate
                for baseline, forecast in forecasts.items():

                    result = evaluate_context_mase(
                        Y_test,
                        forecast,
                        scales
                    )

                    summary_rows.append({
                        "asset_class": asset_class,
                        "dataset": dataset,
                        "baseline": baseline,
                        "rmse": result["rmse"],
                        "mae": result["mae"],
                        "mase_250": result["mase_250"],
                        "mean_context_scale": float(
                            np.mean(scales)
                        ),
                        "min_context_scale": float(
                            np.min(scales)
                        ),
                        "n_origins": len(Y_test),
                    })

                    for h in range(HORIZON):

                        horizon_rows.append({
                            "asset_class": asset_class,
                            "dataset": dataset,
                            "baseline": baseline,
                            "horizon": h + 1,
                            "rmse": result[
                                "rmse_by_horizon"
                            ][h],
                            "mae": result[
                                "mae_by_horizon"
                            ][h],
                            "mase_250": result[
                                "mase_250_by_horizon"
                            ][h],
                        })

                print(
                    f"{asset_class:10s} "
                    f"{dataset:25s} "
                    f"origins={len(Y_test)}"
                )

            except Exception as exc:

                failures.append({
                    "asset_class": asset_class,
                    "dataset": dataset,
                    "error": str(exc),
                })

                print(
                    f"ERROR: {dataset}: {exc}"
                )

    # --------------------------------------------------------
    # SAVE NEW FILES
    # --------------------------------------------------------

    summary_df = pd.DataFrame(
        summary_rows
    )

    horizon_df = pd.DataFrame(
        horizon_rows
    )

    summary_path = (
        BASE_DIR
        / f"point_baselines_context_mase_{frequency}.csv"
    )

    horizon_path = (
        BASE_DIR
        / f"point_baselines_context_mase_by_horizon_{frequency}.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    horizon_df.to_csv(
        horizon_path,
        index=False
    )

    print()
    print(f"Saved: {summary_path}")
    print(f"Saved: {horizon_path}")

    print(
        f"Successful datasets: "
        f"{summary_df[['asset_class', 'dataset']].drop_duplicates().shape[0]}"
    )

    print(f"Failures: {len(failures)}")

    if failures:
        print(pd.DataFrame(failures).to_string(index=False))
        raise RuntimeError(
            f"{frequency}: some datasets failed. "
            "Review errors before using the results."
        )

### Learned models

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

LOOKBACK = 250
HORIZON = 20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ============================================================
# MODEL DIRECTORIES
# ============================================================

POINT_MODEL_DIRS = {
    "ARIMA": "arima_standardized",
    "LightGBM": "lightgbm_standardized_train_test",
    "LSTM": "lstm_standardized_train_test",
    "Sundial": "sundial_standardized_train_test",
    "Lag-Llama": "lag-llama_standardized_train_test",
    "Chronos-T5": "chronos_standardized_train_test",
    "Toto": "toto_standardized_train_test",
    "TimesFM": "timesfm_standardized_train_test",
    "Moirai": "moirai_standardized_train_test",
    "Moirai-MoE": "moirai_moe_standardized_train_test",
}


# ============================================================
# EXACT POINT-FORECAST SUFFIXES
# ============================================================

MODEL_SUFFIXES = {
    "ARIMA": "_arima",
    "LightGBM": "_lightgbm",
    "LSTM": "_lstm",
    "Sundial": "_sundial_zero-shot",
    "Lag-Llama": "_lag-llama_zero-shot",
    "Chronos-T5": "_chronos-t5-small_zero-shot",
    "Toto": "_toto_zero-shot",
    "TimesFM": "_timesfm-2.5-200m_zero-shot",
    "Moirai": "_moirai-2.0-r-small_zero-shot",
    "Moirai-MoE": "_moirai-moe-1.0-r-base_zero-shot",
}


X_COLS = [
    f"X_{i}"
    for i in range(LOOKBACK)
]

Y_COLS = [
    f"Y_{h}"
    for h in range(HORIZON)
]


# ============================================================
# HELPER: LOAD ORIGINAL TEST DATA
# ============================================================

def load_test_xy(test_path):

    df = pd.read_csv(test_path)

    missing = [
        c for c in X_COLS + Y_COLS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {test_path.name}: "
            f"{missing}"
        )

    X = df[X_COLS].to_numpy(dtype=float)
    Y = df[Y_COLS].to_numpy(dtype=float)

    if X.shape[1] != LOOKBACK:
        raise ValueError(
            f"Unexpected X shape: {X.shape}"
        )

    if Y.shape[1] != HORIZON:
        raise ValueError(
            f"Unexpected Y shape: {Y.shape}"
        )

    if not np.all(np.isfinite(X)):
        raise ValueError(
            f"Non-finite X values in {test_path.name}"
        )

    if not np.all(np.isfinite(Y)):
        raise ValueError(
            f"Non-finite Y values in {test_path.name}"
        )

    return X, Y


# ============================================================
# HELPER: ROLLING 250-CONTEXT MASE SCALE
# ============================================================

def calculate_context_scales(X):

    """
    X shape:
        (n_origins, 250)

    Returns:
        One MASE scaling factor per forecast origin.

    scale_i =
        mean(abs(diff(X_i)))

    Shape:
        (n_origins,)
    """

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != LOOKBACK:
        raise ValueError(
            f"Unexpected context shape: {X.shape}"
        )

    differences = np.abs(
        np.diff(X, axis=1)
    )

    scales = np.mean(
        differences,
        axis=1
    )

    if not np.all(np.isfinite(scales)):
        raise ValueError(
            "Non-finite context scales detected."
        )

    if np.any(scales <= 0):
        raise ValueError(
            "Zero or negative context MASE scale."
        )

    return scales


# ============================================================
# HELPER: LOAD SAVED POINT FORECASTS
# ============================================================

def load_point_forecasts(prediction_path):

    df = pd.read_csv(prediction_path)

    missing = [
        c for c in Y_COLS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing prediction columns in "
            f"{prediction_path.name}: {missing}"
        )

    predictions = df[
        Y_COLS
    ].to_numpy(dtype=float)

    if not np.all(np.isfinite(predictions)):
        raise ValueError(
            f"Non-finite predictions in "
            f"{prediction_path.name}"
        )

    return predictions


# ============================================================
# HELPER: EVALUATE ONE MODEL ON ONE DATASET
# ============================================================

def evaluate_point_context_mase(
    Y,
    predictions,
    context_scales,
):

    Y = np.asarray(Y, dtype=float)

    predictions = np.asarray(
        predictions,
        dtype=float
    )

    context_scales = np.asarray(
        context_scales,
        dtype=float
    )

    if Y.shape != predictions.shape:
        raise ValueError(
            f"Prediction shape mismatch: "
            f"Y={Y.shape}, "
            f"predictions={predictions.shape}"
        )

    if context_scales.shape != (Y.shape[0],):
        raise ValueError(
            "Context scales do not match "
            "the number of forecast origins."
        )

    # --------------------------------------------------------
    # ERRORS
    # --------------------------------------------------------

    errors = Y - predictions

    absolute_errors = np.abs(errors)

    squared_errors = errors ** 2


    # --------------------------------------------------------
    # RMSE
    #
    # 1. RMSE separately for each horizon
    # 2. Average horizon RMSE values
    # --------------------------------------------------------

    rmse_by_horizon = np.sqrt(
        np.mean(
            squared_errors,
            axis=0
        )
    )

    overall_rmse = float(
        np.mean(rmse_by_horizon)
    )


    # --------------------------------------------------------
    # MAE
    # --------------------------------------------------------

    mae_by_horizon = np.mean(
        absolute_errors,
        axis=0
    )

    overall_mae = float(
        np.mean(mae_by_horizon)
    )


    # --------------------------------------------------------
    # ROLLING 250-CONTEXT MASE
    #
    # Divide BEFORE averaging.
    # --------------------------------------------------------

    scaled_errors = (
        absolute_errors
        / context_scales[:, None]
    )

    mase_by_horizon = np.mean(
        scaled_errors,
        axis=0
    )

    overall_mase = float(
        np.mean(mase_by_horizon)
    )


    return {
        "rmse": overall_rmse,
        "mae": overall_mae,
        "mase_250": overall_mase,
        "rmse_by_horizon": rmse_by_horizon,
        "mae_by_horizon": mae_by_horizon,
        "mase_250_by_horizon": mase_by_horizon,
    }


# ============================================================
# PROCESS DAILY AND WEEKLY
# ============================================================

all_summary_rows = []
all_horizon_rows = []
all_failures = []

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 75)
    print(f"PROCESSING {frequency.upper()}")
    print("=" * 75)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    if not DATA_DIR.is_dir():
        raise FileNotFoundError(DATA_DIR)

    if not BASE_DIR.is_dir():
        raise FileNotFoundError(BASE_DIR)

    frequency_summary = []
    frequency_horizon = []
    frequency_failures = []

    # --------------------------------------------------------
    # LOOP THROUGH ASSET CLASSES
    # --------------------------------------------------------

    for folder, asset_class in ASSET_ORDER.items():

        test_asset_dir = DATA_DIR / folder

        if not test_asset_dir.is_dir():
            raise FileNotFoundError(test_asset_dir)

        test_files = sorted(
            test_asset_dir.glob("test_*.csv")
        )

        print()
        print(
            f"{asset_class}: "
            f"{len(test_files)} test datasets"
        )

        for test_path in test_files:

            dataset_stem = test_path.stem

            dataset_name = (
                dataset_stem.removeprefix("test_")
            )

            try:

                # ------------------------------------------------
                # ORIGINAL TEST INPUTS AND TARGETS
                # ------------------------------------------------

                X_test, Y_test = load_test_xy(
                    test_path
                )

                n_origins = len(Y_test)

                if n_origins == 0:
                    raise ValueError(
                        "Empty test dataset."
                    )

                # ------------------------------------------------
                # ONE SCALE PER ORIGIN
                # ------------------------------------------------

                context_scales = calculate_context_scales(
                    X_test
                )

            except Exception as exc:

                # All models depend on these shared inputs.
                # Record every affected model.

                for model_name in POINT_MODEL_DIRS:

                    frequency_failures.append({
                        "frequency": frequency,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "model": model_name,
                        "error": str(exc),
                    })

                print(
                    f"  ERROR loading {dataset_name}: {exc}"
                )

                continue


            # ----------------------------------------------------
            # LOOP THROUGH MODELS
            # ----------------------------------------------------

            for model_name, model_dir in POINT_MODEL_DIRS.items():

                model_asset_dir = (
                    BASE_DIR
                    / model_dir
                    / folder
                )

                model_suffix = MODEL_SUFFIXES[
                    model_name
                ]

                prediction_path = (
                    model_asset_dir
                    / f"{dataset_stem}{model_suffix}.csv"
                )

                try:

                    if not prediction_path.is_file():
                        raise FileNotFoundError(
                            prediction_path
                        )

                    # --------------------------------------------
                    # LOAD SAVED POINT FORECASTS
                    # --------------------------------------------

                    Y_pred = load_point_forecasts(
                        prediction_path
                    )

                    # --------------------------------------------
                    # CHECK ALIGNMENT SHAPE
                    # --------------------------------------------

                    if Y_pred.shape != Y_test.shape:
                        raise ValueError(
                            f"Shape mismatch: "
                            f"pred={Y_pred.shape}, "
                            f"actual={Y_test.shape}"
                        )

                    # --------------------------------------------
                    # EVALUATE
                    # --------------------------------------------

                    result = evaluate_point_context_mase(
                        Y_test,
                        Y_pred,
                        context_scales
                    )

                    # --------------------------------------------
                    # SUMMARY ROW
                    # --------------------------------------------

                    frequency_summary.append({
                        "frequency": frequency,
                        "model": model_name,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "rmse": result["rmse"],
                        "mae": result["mae"],
                        "mase_250": result["mase_250"],
                        "n_origins": n_origins,
                        "mean_context_scale": float(
                            np.mean(context_scales)
                        ),
                        "min_context_scale": float(
                            np.min(context_scales)
                        ),
                    })

                    # --------------------------------------------
                    # HORIZON ROWS
                    # --------------------------------------------

                    for h in range(HORIZON):

                        frequency_horizon.append({
                            "frequency": frequency,
                            "model": model_name,
                            "asset_class": asset_class,
                            "dataset": dataset_name,
                            "horizon": h + 1,
                            "rmse": float(
                                result["rmse_by_horizon"][h]
                            ),
                            "mae": float(
                                result["mae_by_horizon"][h]
                            ),
                            "mase_250": float(
                                result["mase_250_by_horizon"][h]
                            ),
                        })

                except Exception as exc:

                    frequency_failures.append({
                        "frequency": frequency,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "model": model_name,
                        "error": str(exc),
                    })

                    print(
                        f"  ERROR: {model_name} / "
                        f"{dataset_name}: {exc}"
                    )


    # ========================================================
    # SUMMARY DATAFRAMES
    # ========================================================

    summary_df = pd.DataFrame(
        frequency_summary
    )

    horizon_df = pd.DataFrame(
        frequency_horizon
    )

    failures_df = pd.DataFrame(
        frequency_failures
    )

    # --------------------------------------------------------
    # EXPECTED COVERAGE
    # --------------------------------------------------------

    n_datasets = sum(
        len(list((DATA_DIR / folder).glob("test_*.csv")))
        for folder in ASSET_ORDER
    )

    n_models = len(POINT_MODEL_DIRS)

    expected_rows = n_datasets * n_models

    print()
    print("-" * 75)
    print(f"Frequency: {frequency}")
    print(f"Datasets found: {n_datasets}")
    print(f"Models: {n_models}")
    print(f"Expected model-dataset rows: {expected_rows}")
    print(f"Successful rows: {len(summary_df)}")
    print(f"Failed rows: {len(failures_df)}")
    print("-" * 75)

    if not summary_df.empty:

        print()
        print("Mean MASE by model:")

        print(
            summary_df
            .groupby("model")["mase_250"]
            .agg(["count", "mean", "median"])
            .sort_values("mean")
            .round(6)
            .to_string()
        )


    # ========================================================
    # SAVE
    #
    # Do not overwrite original metric files.
    # ========================================================

    summary_path = (
        BASE_DIR
        / f"point_models_context_mase_{frequency}.csv"
    )

    horizon_path = (
        BASE_DIR
        / f"point_models_context_mase_by_horizon_{frequency}.csv"
    )

    failures_path = (
        BASE_DIR
        / f"point_models_context_mase_failures_{frequency}.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    horizon_df.to_csv(
        horizon_path,
        index=False
    )

    failures_df.to_csv(
        failures_path,
        index=False
    )

    print()
    print(f"Saved: {summary_path}")
    print(f"Saved: {horizon_path}")
    print(f"Saved: {failures_path}")

    all_summary_rows.extend(frequency_summary)
    all_horizon_rows.extend(frequency_horizon)
    all_failures.extend(frequency_failures)

    if len(failures_df) > 0:
        print(
            "\nWARNING: Some model-dataset combinations "
            "failed. Check the failures CSV."
        )

    if len(summary_df) != expected_rows:
        print(
            "\nWARNING: Coverage is incomplete."
        )


# ============================================================
# FINAL COVERAGE CHECK
# ============================================================

print()
print("=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

all_summary_df = pd.DataFrame(
    all_summary_rows
)

if not all_summary_df.empty:

    print(
        all_summary_df
        .groupby(["frequency", "model"])["mase_250"]
        .agg(["count", "mean", "median"])
        .round(6)
        .to_string()
    )

print()
print(f"Total failures: {len(all_failures)}")

if all_failures:
    print(
        "\nReview the saved failure reports."
    )

### Evaluation

Calculate:

- Mean MASE across datasets.
- Median MASE across datasets.
- Mean within-dataset rank.
- Number of datasets won.
- Fraction of datasets on which each method beats zero.
- Mean RMSE across datasets.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

for frequency in ["daily", "weekly"]:

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    baseline_path = (
        BASE_DIR
        / f"point_baselines_context_mase_{frequency}.csv"
    )

    model_path = (
        BASE_DIR
        / f"point_models_context_mase_{frequency}.csv"
    )

    baselines = pd.read_csv(baseline_path)
    models = pd.read_csv(model_path)

    # Standardize method column
    baselines = baselines.rename(
        columns={"baseline": "model"}
    )

    baselines["type"] = "Simple baseline"
    models["type"] = "Forecasting model"

    required = [
        "asset_class",
        "dataset",
        "model",
        "rmse",
        "mae",
        "mase_250",
    ]

    for name, df in [
        ("Baselines", baselines),
        ("Models", models),
    ]:
        missing = [
            col for col in required
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"{name} missing columns: {missing}"
            )

    combined = pd.concat(
        [
            baselines[
                required + ["type"]
            ],
            models[
                required + ["type"]
            ],
        ],
        ignore_index=True,
    )

    # Prevent duplicate dataset-method entries
    keys = [
        "asset_class",
        "dataset",
        "model",
    ]

    if combined.duplicated(keys).any():
        raise ValueError(
            "Duplicate dataset-method entries detected."
        )

    dataset_keys = [
        "asset_class",
        "dataset",
    ]

    # Ensure every method has the same dataset coverage
    counts = combined.groupby("model").size()

    n_datasets = (
        combined[dataset_keys]
        .drop_duplicates()
        .shape[0]
    )

    if not (counts == n_datasets).all():
        raise ValueError(
            "Incomplete model coverage:\n"
            + counts.to_string()
        )

    # --------------------------------------------------
    # RANK MODELS WITHIN EACH DATASET
    # --------------------------------------------------

    combined["mase_rank"] = (
        combined
        .groupby(dataset_keys)["mase_250"]
        .rank(method="average", ascending=True)
    )

    combined["rmse_rank"] = (
        combined
        .groupby(dataset_keys)["rmse"]
        .rank(method="average", ascending=True)
    )

    # --------------------------------------------------
    # COMPARE AGAINST ZERO RETURN
    # --------------------------------------------------

    zero = (
        combined[
            combined["model"] == "Zero"
        ][dataset_keys + ["mase_250", "rmse"]]
        .rename(
            columns={
                "mase_250": "zero_mase",
                "rmse": "zero_rmse",
            }
        )
    )

    combined = combined.merge(
        zero,
        on=dataset_keys,
        how="left",
        validate="many_to_one",
    )

    if combined[
        ["zero_mase", "zero_rmse"]
    ].isna().any().any():
        raise ValueError(
            "Some datasets are missing the zero baseline."
        )

    combined["beats_zero_mase"] = (
        combined["mase_250"]
        <
        combined["zero_mase"]
    )

    combined["beats_zero_rmse"] = (
        combined["rmse"]
        <
        combined["zero_rmse"]
    )

    # --------------------------------------------------
    # WINS
    # --------------------------------------------------

    # Count all methods tied for best score.
    # Wins may therefore sum to more than D.
    best_mase = (
        combined
        .groupby(dataset_keys)["mase_250"]
        .transform("min")
    )

    combined["mase_win"] = (
        combined["mase_250"] == best_mase
    )

    # --------------------------------------------------
    # SUMMARIZE
    # --------------------------------------------------

    summary = (
        combined
        .groupby(["model", "type"])
        .agg(
            datasets=("dataset", "size"),
            mean_mase=("mase_250", "mean"),
            median_mase=("mase_250", "median"),
            mean_mase_rank=("mase_rank", "mean"),
            mase_wins=("mase_win", "sum"),
            beats_zero_mase_pct=(
                "beats_zero_mase", "mean"
            ),
            mean_rmse=("rmse", "mean"),
            mean_rmse_rank=("rmse_rank", "mean"),
            beats_zero_rmse_pct=(
                "beats_zero_rmse", "mean"
            ),
        )
        .reset_index()
    )

    summary[
        "beats_zero_mase_pct"
    ] *= 100

    summary[
        "beats_zero_rmse_pct"
    ] *= 100

    summary = summary.sort_values(
        "mean_mase_rank"
    )

    print()
    print("=" * 80)
    print(f"{frequency.upper()} COMPARISON")
    print("=" * 80)

    print(
        summary.round(4).to_string(index=False)
    )

    combined.to_csv(
        BASE_DIR
        / f"point_all_context_mase_{frequency}.csv",
        index=False,
    )

    summary.to_csv(
        BASE_DIR
        / f"point_all_context_mase_summary_{frequency}.csv",
        index=False,
    )

### Check magnitude of location estimates

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

LOOKBACK = 250
HORIZON = 20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

rows = []

for frequency in ["daily", "weekly"]:

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    for folder, asset_class in ASSET_ORDER.items():

        for test_path in sorted(
            (DATA_DIR / folder).glob("test_*.csv")
        ):

            dataset = test_path.stem.removeprefix("test_")

            train_path = (
                DATA_DIR
                / folder
                / f"train_{dataset}.csv"
            )

            if not train_path.exists():
                raise FileNotFoundError(train_path)

            X, Y = load_xy(test_path)

            train_series = reconstruct_training_series(
                train_path
            )

            train_mean = float(np.mean(train_series))
            train_median = float(np.median(train_series))

            context_means = np.mean(X, axis=1)
            context_medians = np.median(X, axis=1)

            typical_abs_return = float(
                np.mean(np.abs(Y))
            )

            if typical_abs_return <= 0:
                raise ValueError(
                    f"Nonpositive return magnitude: {dataset}"
                )

            rows.append({
                "frequency": frequency,
                "asset_class": asset_class,
                "dataset": dataset,

                "train_mean": train_mean,
                "train_median": train_median,

                "abs_train_mean": abs(train_mean),
                "abs_train_median": abs(train_median),

                "mean_abs_context_mean": float(
                    np.mean(np.abs(context_means))
                ),

                "mean_abs_context_median": float(
                    np.mean(np.abs(context_medians))
                ),

                "typical_abs_return": typical_abs_return,

                "train_mean_ratio": (
                    abs(train_mean) / typical_abs_return
                ),

                "train_median_ratio": (
                    abs(train_median) / typical_abs_return
                ),

                "context_mean_ratio": (
                    np.mean(np.abs(context_means))
                    / typical_abs_return
                ),

                "context_median_ratio": (
                    np.mean(np.abs(context_medians))
                    / typical_abs_return
                ),
            })

location_df = pd.DataFrame(rows)

for frequency in ["daily", "weekly"]:

    freq_df = location_df[
        location_df["frequency"] == frequency
    ]

    print()
    print("=" * 75)
    print(f"{frequency.upper()} LOCATION MAGNITUDE")
    print("=" * 75)

    cols = [
        "train_mean_ratio",
        "train_median_ratio",
        "context_mean_ratio",
        "context_median_ratio",
    ]

    print(
        freq_df[cols]
        .agg(["mean", "median", "min", "max"])
        .round(5)
        .to_string()
    )

    output_path = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
        / f"location_magnitude_{frequency}.csv"
    )

    freq_df.to_csv(
        output_path,
        index=False
    )

    print(f"\nSaved: {output_path}")

### Checks

- How often do the fixed location baselines outperform their rolling counterparts?

- How much do learned models improve or deteriorate relative to zero?

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

for frequency in ["daily", "weekly"]:

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    path = (
        BASE_DIR
        / f"point_all_context_mase_{frequency}.csv"
    )

    df = pd.read_csv(path)

    keys = ["asset_class", "dataset"]

    # ---------------------------------------------------
    # A. FIXED VS ROLLING LOCATION BASELINES
    # ---------------------------------------------------

    wide = df.pivot(
        index=keys,
        columns="model",
        values="mase_250"
    )

    comparisons = [
        ("Train Mean", "Context Mean"),
        ("Train Median", "Context Median"),
    ]

    location_rows = []

    for fixed, rolling in comparisons:

        difference = (
            wide[rolling] - wide[fixed]
        )

        location_rows.append({
            "comparison": f"{fixed} vs {rolling}",
            "datasets": len(difference),
            "fixed_better_pct": (
                100 * (difference > 0).mean()
            ),
            "rolling_better_pct": (
                100 * (difference < 0).mean()
            ),
            "mean_rolling_minus_fixed": (
                difference.mean()
            ),
            "median_rolling_minus_fixed": (
                difference.median()
            ),
        })

    location_summary = pd.DataFrame(
        location_rows
    )

    print()
    print("=" * 75)
    print(f"{frequency.upper()}: FIXED VS ROLLING")
    print("=" * 75)

    print(
        location_summary
        .round(6)
        .to_string(index=False)
    )

    # ---------------------------------------------------
    # B. RELATIVE IMPROVEMENT OVER ZERO
    # ---------------------------------------------------

    zero = wide["Zero"]

    relative_improvement = (
        wide.sub(zero, axis=0)
        .mul(-1)
        .div(zero, axis=0)
    ) * 100

    improvement_summary = (
        relative_improvement
        .agg(["mean", "median"])
        .T
        .rename(
            columns={
                "mean": "mean_improvement_pct",
                "median": "median_improvement_pct",
            }
        )
    )

    improvement_summary[
        "beats_zero_pct"
    ] = (
        relative_improvement.gt(0).mean() * 100
    )

    improvement_summary = (
        improvement_summary
        .sort_values(
            "median_improvement_pct",
            ascending=False
        )
        .reset_index()
        .rename(columns={"model": "model"})
    )

    print()
    print("=" * 75)
    print(
        f"{frequency.upper()}: "
        "RELATIVE IMPROVEMENT OVER ZERO"
    )
    print("=" * 75)

    print(
        improvement_summary
        .round(4)
        .to_string(index=False)
    )

    # ---------------------------------------------------
    # SAVE
    # ---------------------------------------------------

    location_summary.to_csv(
        BASE_DIR
        / f"fixed_vs_rolling_location_{frequency}.csv",
        index=False
    )

    improvement_summary.to_csv(
        BASE_DIR
        / f"mase_improvement_over_zero_{frequency}.csv",
        index=False
    )